# Handwritten digits & math symbols - walkthrough

Runs the same pipeline as `python -m src.train`, step by step.

In [ ]:
import os, sys
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.getcwd())

import numpy as np
import matplotlib.pyplot as plt

from src.data import load_dataset, split_data
from src.augment import AUGMENTATIONS, augment_dataset, augment_once
from src.models import build_cnn, make_callbacks, gradcam_heatmap, overlay_heatmap

DATA_DIR = 'data/handwritten_dataset'   # see README: python scripts/download_data.py --url ...
SEED = 42

## 1. Load data and split

Train / val / test are split *before* any augmentation. Validation is used for EarlyStopping; test is used once at the end.

In [ ]:
X, y, class_names, data_stats = load_dataset(DATA_DIR)
print(data_stats)
parts = split_data(X, y, seed=SEED)
(X_tr, y_tr), (X_va, y_va), (X_te, y_te) = parts['train'], parts['val'], parts['test']
print(class_names)
print('train', X_tr.shape, 'val', X_va.shape, 'test', X_te.shape)

fig, axes = plt.subplots(2, 8, figsize=(12, 3.4))
for ax, i in zip(axes.ravel(), np.random.default_rng(0).choice(len(X_tr), 16, replace=False)):
    ax.imshow(X_tr[i].squeeze(), cmap='gray'); ax.set_title(class_names[y_tr[i]], fontsize=9); ax.axis('off')
plt.show()

## 2. Augmentation

Three families: stroke (rotation / pen thickness / blur), occlusion, photometric (brightness, noise, shift).

In [ ]:
rng = np.random.default_rng(0)
fig, axes = plt.subplots(3, 7, figsize=(12, 5.4))
for r in range(3):
    img = X_tr[np.where(y_tr == r)[0][0]]
    axes[r, 0].imshow(img.squeeze(), cmap='gray'); axes[r, 0].set_title(f'{class_names[r]} (orig)', fontsize=8)
    c = 1
    for name, fn in AUGMENTATIONS.items():
        for _ in range(2):
            axes[r, c].imshow(fn(img, rng).squeeze(), cmap='gray'); axes[r, c].set_title(name, fontsize=8); c += 1
for ax in axes.ravel(): ax.axis('off')
plt.show()

In [ ]:
X_ext, y_ext, counts = augment_dataset(X_tr, y_tr, per_image=2, seed=SEED)
X_te_aug = augment_once(X_te, seed=SEED + 1)
print('original train:', len(X_tr), '-> extended:', len(X_ext), counts)

## 3. Baseline: HOG + linear SVM

In [ ]:
from skimage.feature import hog
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

def hog_feats(A):
    return np.array([hog(a.squeeze(), pixels_per_cell=(8, 8), cells_per_block=(2, 2)) for a in A])

svm = LinearSVC(dual='auto').fit(hog_feats(X_tr), y_tr)
svm_pred = svm.predict(hog_feats(X_te))
print('SVM clean test  acc', accuracy_score(y_te, svm_pred), 'F1', f1_score(y_te, svm_pred, average='macro'))

## 4. CNN on real data vs. CNN on real + augmented data

Same architecture, same callbacks, same validation split.

In [ ]:
results = {}
models = {}
for tag, (Xf, yf) in {'CNN (real only)': (X_tr, y_tr), 'CNN (augmented)': (X_ext, y_ext)}.items():
    m = build_cnn(len(class_names), X_tr.shape[1:])
    m.fit(Xf, yf, validation_data=(X_va, y_va), epochs=30, batch_size=64, callbacks=make_callbacks(), verbose=2)
    models[tag] = m
    for split, Xe in [('clean', X_te), ('augmented', X_te_aug)]:
        p = np.argmax(m.predict(Xe, verbose=0), 1)
        results[(tag, split)] = (accuracy_score(y_te, p), f1_score(y_te, p, average='macro'))
for k, (a, f) in results.items():
    print(f'{k[0]:18s} {k[1]:10s} acc={a:.4f} F1={f:.4f}')

## 5. Confusion matrix and Grad-CAM (augmented CNN)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix
best = models['CNN (augmented)']
pred = np.argmax(best.predict(X_te, verbose=0), 1)
fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay(confusion_matrix(y_te, pred), display_labels=class_names).plot(ax=ax, cmap='Blues', colorbar=False)
plt.show()

idx = np.random.default_rng(0).choice(len(X_te), 6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(12, 4.4))
for j, i in enumerate(idx):
    heat = gradcam_heatmap(best, X_te[i], int(pred[i]))
    axes[0, j].imshow(X_te[i].squeeze(), cmap='gray'); axes[0, j].set_title(f'true {class_names[y_te[i]]}', fontsize=8)
    axes[1, j].imshow(overlay_heatmap(X_te[i], heat)); axes[1, j].set_title(f'pred {class_names[pred[i]]}', fontsize=8)
for ax in axes.ravel(): ax.axis('off')
plt.show()

## 6. Demo

Train with `python -m src.train` to save the models, then run `python app.py`.